# Case 2: Airbnb Rental Arbitrage

## Analytics Application for Joan Murray

This notebook develops an interactive Gradio application that helps Joan Murray evaluate Airbnb rental-arbitrage opportunities in New York City.

The application will:

- Compare nightly prices across boroughs
- Examine reviews and nightly prices
- Examine minimum-night requirements and nightly prices
- Examine annual availability and nightly prices
- Evaluate a Multiple Linear Regression model
- Compare training and testing performance using an 80/20 split
- Estimate the nightly price of a potential listing
- Identify whether the predicted price is above $120

In [ ]:
!pip install -q gradio

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

In [ ]:
uploaded = files.upload()

In [ ]:
possible_files = [
    "Airbnb_db.csv",
    "Airbnn_db.csv"
]

FILE_NAME = None

for file_name in possible_files:
    if os.path.exists(file_name):
        FILE_NAME = file_name
        break

if FILE_NAME is None:
    raise FileNotFoundError(
        "Please upload Airbnb_db.csv before running this notebook."
    )

df = pd.read_csv(FILE_NAME)

print(f"Dataset loaded successfully: {FILE_NAME}")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe().round(2))

In [ ]:
required_columns = [
    "id",
    "neighbourhood_group_c",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "price"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

airbnb = df[required_columns].copy()

numeric_columns = [
    "neighbourhood_group_c",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "price"
]

for column in numeric_columns:
    airbnb[column] = pd.to_numeric(
        airbnb[column],
        errors="coerce"
    )

rows_before = len(airbnb)

airbnb = airbnb.dropna()
airbnb = airbnb.drop_duplicates()

airbnb = airbnb[
    airbnb["neighbourhood_group_c"].between(1, 5)
]

airbnb = airbnb[
    (airbnb["minimum_nights"] >= 1) &
    (airbnb["number_of_reviews"] >= 0) &
    (airbnb["reviews_per_month"] >= 0) &
    (airbnb["availability_365"].between(0, 365)) &
    (airbnb["price"] > 0)
]

rows_after = len(airbnb)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)

In [ ]:
borough_map = {
    1: "Manhattan",
    2: "Brooklyn",
    3: "Queens",
    4: "Staten Island",
    5: "Bronx"
}

airbnb["borough"] = airbnb[
    "neighbourhood_group_c"
].map(borough_map)

airbnb["above_120"] = (
    airbnb["price"] > 120
)

airbnb["price_category"] = np.where(
    airbnb["above_120"],
    "Above $120",
    "$120 or Below"
)

display(airbnb.head())

In [ ]:
borough_summary = (
    airbnb
    .groupby("borough")
    .agg(
        Listings=("price", "size"),
        Average_Price=("price", "mean"),
        Median_Price=("price", "median"),
        Percent_Above_120=(
            "above_120",
            lambda values: values.mean() * 100
        )
    )
    .reset_index()
    .sort_values(
        "Average_Price",
        ascending=False
    )
)

total_listings = len(airbnb)
average_price = airbnb["price"].mean()
median_price = airbnb["price"].median()
percent_above_120 = airbnb["above_120"].mean() * 100
highest_price_borough = borough_summary.iloc[0]["borough"]

print(f"Total listings: {total_listings:,}")
print(f"Average nightly price: ${average_price:,.2f}")
print(f"Median nightly price: ${median_price:,.2f}")
print(f"Listings above $120: {percent_above_120:.1f}%")
print(f"Highest-price borough: {highest_price_borough}")

display(borough_summary.round(2))

In [ ]:
model_data = airbnb[
    [
        "borough",
        "minimum_nights",
        "number_of_reviews",
        "reviews_per_month",
        "availability_365",
        "price"
    ]
].copy()

In [ ]:
borough_order = [
    "Bronx",
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Staten Island"
]

model_data["borough"] = pd.Categorical(
    model_data["borough"],
    categories=borough_order
)

borough_dummies = pd.get_dummies(
    model_data["borough"],
    prefix="borough",
    drop_first=True,
    dtype=int
)

model_data_encoded = pd.concat(
    [
        model_data.drop(columns=["borough"]),
        borough_dummies
    ],
    axis=1
)

X = model_data_encoded.drop(
    columns=["price"]
)

y = model_data_encoded["price"]

print("Predictors:")
print(X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print(
    "Training percentage:",
    round(len(X_train) / len(X) * 100, 1)
)

print(
    "Testing percentage:",
    round(len(X_test) / len(X) * 100, 1)
)

In [ ]:
model = LinearRegression()

model.fit(
    X_train,
    y_train
)

train_predictions = model.predict(
    X_train
)

test_predictions = model.predict(
    X_test
)

print("Multiple Linear Regression model trained successfully.")

In [ ]:
def adjusted_r2(r2_value, n, p):
    if n - p - 1 <= 0:
        return np.nan

    return 1 - (
        (1 - r2_value) *
        (n - 1) /
        (n - p - 1)
    )


train_r2 = r2_score(
    y_train,
    train_predictions
)

test_r2 = r2_score(
    y_test,
    test_predictions
)

train_adj_r2 = adjusted_r2(
    train_r2,
    len(y_train),
    X_train.shape[1]
)

test_adj_r2 = adjusted_r2(
    test_r2,
    len(y_test),
    X_test.shape[1]
)

train_mae = mean_absolute_error(
    y_train,
    train_predictions
)

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        train_predictions
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

metrics_table = pd.DataFrame({
    "Metric": [
        "R²",
        "Adjusted R²",
        "MAE",
        "RMSE"
    ],
    "Training": [
        train_r2,
        train_adj_r2,
        train_mae,
        train_rmse
    ],
    "Testing": [
        test_r2,
        test_adj_r2,
        test_mae,
        test_rmse
    ]
})

display(metrics_table.round(4))

In [ ]:
adjusted_r2_gap = abs(
    train_adj_r2 -
    test_adj_r2
)

if adjusted_r2_gap > 0.10:
    overfitting_assessment = (
        "Potential overfitting is present because the "
        "training and testing Adjusted R² values differ substantially."
    )

elif adjusted_r2_gap > 0.05:
    overfitting_assessment = (
        "The model may show mild overfitting because the "
        "training and testing Adjusted R² values differ moderately."
    )

else:
    overfitting_assessment = (
        "There is no meaningful evidence of overfitting. "
        "The training and testing Adjusted R² values are close."
    )

print(
    f"Adjusted R² gap: {adjusted_r2_gap:.4f}"
)

print(overfitting_assessment)

In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

intercept_row = pd.DataFrame({
    "Feature": ["Intercept"],
    "Coefficient": [model.intercept_]
})

coefficient_table = pd.concat(
    [
        intercept_row,
        coefficient_table
    ],
    ignore_index=True
)

display(coefficient_table.round(4))

In [ ]:
coef_lookup = dict(
    zip(
        X.columns,
        model.coef_
    )
)

coefficient_interpretation = f"""
Holding all other variables constant:

• One additional minimum night is associated with a
  ${coef_lookup['minimum_nights']:.2f} change in expected nightly price.

• One additional customer review is associated with a
  ${coef_lookup['number_of_reviews']:.2f} change in expected nightly price.

• One additional review per month is associated with a
  ${coef_lookup['reviews_per_month']:.2f} change in expected nightly price.

• One additional available booking day per year is associated with a
  ${coef_lookup['availability_365']:.2f} change in expected nightly price.

• Manhattan listings are predicted to be
  ${coef_lookup['borough_Manhattan']:.2f} higher than comparable Bronx listings.

• Brooklyn listings are predicted to be
  ${coef_lookup['borough_Brooklyn']:.2f} higher than comparable Bronx listings.

• Queens listings are predicted to be
  ${coef_lookup['borough_Queens']:.2f} higher than comparable Bronx listings.

• Staten Island listings are predicted to be
  ${coef_lookup['borough_Staten Island']:.2f} higher than comparable Bronx listings.
"""

print(coefficient_interpretation)

In [ ]:
def filter_data(borough_filter, min_nights_range):
    filtered = airbnb.copy()

    if borough_filter != "All":
        filtered = filtered[
            filtered["borough"] == borough_filter
        ]

    filtered = filtered[
        filtered["minimum_nights"].between(
            min_nights_range[0],
            min_nights_range[1]
        )
    ]

    return filtered


def plot_price_by_borough(borough_filter, min_nights_range):
    filtered = filter_data(
        borough_filter,
        min_nights_range
    )

    summary = (
        filtered
        .groupby("borough")["price"]
        .mean()
        .sort_values(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(7, 4))

    summary.plot(
        kind="bar",
        ax=ax
    )

    ax.set_title("Average Nightly Price by Borough")
    ax.set_xlabel("Borough")
    ax.set_ylabel("Average Price ($)")
    ax.tick_params(axis="x", rotation=30)

    fig.tight_layout()

    return fig


def plot_percent_above_120(borough_filter, min_nights_range):
    filtered = filter_data(
        borough_filter,
        min_nights_range
    )

    summary = (
        filtered
        .groupby("borough")["above_120"]
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(7, 4))

    summary.plot(
        kind="bar",
        ax=ax
    )

    ax.set_title("Listings Above $120 by Borough")
    ax.set_xlabel("Borough")
    ax.set_ylabel("Percentage Above $120")
    ax.tick_params(axis="x", rotation=30)

    fig.tight_layout()

    return fig

In [ ]:
def plot_reviews(borough_filter, min_nights_range):
    filtered = filter_data(
        borough_filter,
        min_nights_range
    )

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.scatter(
        filtered["reviews_per_month"],
        filtered["price"],
        alpha=0.35
    )

    ax.set_title("Reviews per Month vs Nightly Price")
    ax.set_xlabel("Reviews per Month")
    ax.set_ylabel("Nightly Price ($)")

    fig.tight_layout()

    return fig


def plot_minimum_nights(borough_filter, min_nights_range):
    filtered = filter_data(
        borough_filter,
        min_nights_range
    )

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.scatter(
        filtered["minimum_nights"],
        filtered["price"],
        alpha=0.35
    )

    ax.set_title("Minimum Nights vs Nightly Price")
    ax.set_xlabel("Minimum Nights")
    ax.set_ylabel("Nightly Price ($)")

    fig.tight_layout()

    return fig


def plot_availability(borough_filter, min_nights_range):
    filtered = filter_data(
        borough_filter,
        min_nights_range
    )

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.scatter(
        filtered["availability_365"],
        filtered["price"],
        alpha=0.35
    )

    ax.set_title("Availability vs Nightly Price")
    ax.set_xlabel("Availability per Year")
    ax.set_ylabel("Nightly Price ($)")

    fig.tight_layout()

    return fig

In [ ]:
def make_kpis(borough_filter, min_nights_range):
    filtered = filter_data(
        borough_filter,
        min_nights_range
    )

    if len(filtered) == 0:
        return "No listings match the selected filters."

    average_filtered_price = filtered["price"].mean()
    average_availability = filtered["availability_365"].mean()
    filtered_above_120 = filtered["above_120"].mean() * 100

    return f"""
### Filtered Listing Summary

- **Listings:** {len(filtered):,}
- **Average nightly price:** ${average_filtered_price:,.2f}
- **Average availability:** {average_availability:,.0f} days per year
- **Listings above $120:** {filtered_above_120:.1f}%
"""

In [ ]:
def predict_price(
    borough,
    minimum_nights,
    number_of_reviews,
    reviews_per_month,
    availability
):
    minimum_nights = max(
        1,
        float(minimum_nights)
    )

    number_of_reviews = max(
        0,
        float(number_of_reviews)
    )

    reviews_per_month = max(
        0,
        float(reviews_per_month)
    )

    availability = min(
        365,
        max(0, float(availability))
    )

    row = {
        column: 0.0
        for column in X.columns
    }

    row["minimum_nights"] = minimum_nights
    row["number_of_reviews"] = number_of_reviews
    row["reviews_per_month"] = reviews_per_month
    row["availability_365"] = availability

    borough_column = f"borough_{borough}"

    if borough_column in row:
        row[borough_column] = 1.0

    input_data = pd.DataFrame(
        [row]
    )[X.columns]

    predicted_price = model.predict(
        input_data
    )[0]

    if predicted_price > 120:
        classification = "Above $120"
        conclusion = (
            "This listing meets the case threshold for a "
            "potentially higher-value opportunity."
        )
    else:
        classification = "$120 or Below"
        conclusion = (
            "This listing does not meet the case threshold "
            "for a higher-value opportunity."
        )

    return f"""
## Estimated Nightly Price

### ${predicted_price:,.2f}

**Price classification:** {classification}

{conclusion}

This estimate should be reviewed together with monthly rent, occupancy, cleaning costs, platform fees, local regulations, and written landlord permission.
"""

In [ ]:
overview_text = f"""
## Executive Overview

This application analyzes {total_listings:,} Airbnb listings in New York City.

- **Average nightly price:** ${average_price:,.2f}
- **Median nightly price:** ${median_price:,.2f}
- **Listings above $120:** {percent_above_120:.1f}%
- **Highest average-price borough:** {highest_price_borough}
"""

reliability_text = f"""
## Model Reliability

The Multiple Linear Regression model was evaluated using an
**80% training and 20% testing split**.

| Metric | Training | Testing |
|---|---:|---:|
| R² | {train_r2:.4f} | {test_r2:.4f} |
| Adjusted R² | {train_adj_r2:.4f} | {test_adj_r2:.4f} |
| MAE | ${train_mae:,.2f} | ${test_mae:,.2f} |
| RMSE | ${train_rmse:,.2f} | ${test_rmse:,.2f} |

**Adjusted R² gap:** {adjusted_r2_gap:.4f}

**Overfitting assessment:** {overfitting_assessment}

The model should be used as a decision-support tool rather than a guarantee of future rental income.
"""

In [ ]:
borough_choices = [
    "All",
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Staten Island",
    "Bronx"
]

In [ ]:
# ---------------------------------------------------------
# GRADIO APPLICATION
# ---------------------------------------------------------

simulator_borough_choices = [
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Staten Island",
    "Bronx"
]


# These wrappers combine the two separate sliders into one range list
def update_kpis_from_sliders(
    borough,
    minimum_nights_low,
    minimum_nights_high
):
    return make_kpis(
        borough,
        [minimum_nights_low, minimum_nights_high]
    )


def update_all_plots(
    borough,
    minimum_nights_low,
    minimum_nights_high
):
    nights_range = [
        minimum_nights_low,
        minimum_nights_high
    ]

    return (
        plot_price_by_borough(
            borough,
            nights_range
        ),
        plot_percent_above_120(
            borough,
            nights_range
        ),
        plot_reviews(
            borough,
            nights_range
        ),
        plot_minimum_nights(
            borough,
            nights_range
        ),
        plot_availability(
            borough,
            nights_range
        )
    )


with gr.Blocks(
    title="Airbnb Rental Arbitrage Analytics"
) as app:

    gr.Markdown(
        """
        # Airbnb Rental Arbitrage Analytics

        A decision-support application for Joan Murray.
        """
    )

    # TAB 1: OVERVIEW
    with gr.Tab("Overview"):

        gr.Markdown(overview_text)

        gr.Dataframe(
            value=borough_summary.round(2),
            label="Borough Summary",
            interactive=False
        )

        gr.Markdown(
            """
            Use the filters below to examine a selected group of listings.
            """
        )

        overview_borough = gr.Dropdown(
            choices=borough_choices,
            value="All",
            label="Borough"
        )

        with gr.Row():

            overview_nights_low = gr.Slider(
                minimum=int(
                    airbnb["minimum_nights"].min()
                ),
                maximum=int(
                    airbnb["minimum_nights"].max()
                ),
                value=int(
                    airbnb["minimum_nights"].min()
                ),
                step=1,
                label="Minimum Nights From"
            )

            overview_nights_high = gr.Slider(
                minimum=int(
                    airbnb["minimum_nights"].min()
                ),
                maximum=int(
                    airbnb["minimum_nights"].max()
                ),
                value=int(
                    airbnb["minimum_nights"].max()
                ),
                step=1,
                label="Minimum Nights To"
            )

        overview_button = gr.Button(
            "Update Summary",
            variant="primary"
        )

        overview_output = gr.Markdown()

        overview_button.click(
            fn=update_kpis_from_sliders,
            inputs=[
                overview_borough,
                overview_nights_low,
                overview_nights_high
            ],
            outputs=overview_output
        )

    # TAB 2: EXPLORE DATA
    with gr.Tab("Explore the Data"):

        gr.Markdown(
            """
            ## Interactive Data Exploration

            Select a borough and minimum-night range, then click
            **Update Charts**.
            """
        )

        explore_borough = gr.Dropdown(
            choices=borough_choices,
            value="All",
            label="Borough"
        )

        with gr.Row():

            explore_nights_low = gr.Slider(
                minimum=int(
                    airbnb["minimum_nights"].min()
                ),
                maximum=int(
                    airbnb["minimum_nights"].max()
                ),
                value=int(
                    airbnb["minimum_nights"].min()
                ),
                step=1,
                label="Minimum Nights From"
            )

            explore_nights_high = gr.Slider(
                minimum=int(
                    airbnb["minimum_nights"].min()
                ),
                maximum=int(
                    airbnb["minimum_nights"].max()
                ),
                value=int(
                    airbnb["minimum_nights"].max()
                ),
                step=1,
                label="Minimum Nights To"
            )

        update_charts_button = gr.Button(
            "Update Charts",
            variant="primary"
        )

        with gr.Row():

            borough_price_plot = gr.Plot(
                label="Average Price by Borough"
            )

            above_120_plot = gr.Plot(
                label="Listings Above $120"
            )

        with gr.Row():

            reviews_plot = gr.Plot(
                label="Reviews and Price"
            )

            minimum_nights_plot = gr.Plot(
                label="Minimum Nights and Price"
            )

        availability_plot = gr.Plot(
            label="Availability and Price"
        )

        update_charts_button.click(
            fn=update_all_plots,
            inputs=[
                explore_borough,
                explore_nights_low,
                explore_nights_high
            ],
            outputs=[
                borough_price_plot,
                above_120_plot,
                reviews_plot,
                minimum_nights_plot,
                availability_plot
            ]
        )

    # TAB 3: MODEL PERFORMANCE
    with gr.Tab("Model Performance"):

        gr.Markdown(reliability_text)

        gr.Dataframe(
            value=metrics_table.round(4),
            label="Training and Testing Metrics",
            interactive=False
        )

        gr.Markdown(
            f"""
            ### Reliability Conclusion

            The model used an **80% training and 20% testing split**.

            The difference between the training and testing Adjusted R²
            values is **{adjusted_r2_gap:.4f}**.

            **{overfitting_assessment}**
            """
        )

    # TAB 4: COEFFICIENT INTERPRETATION
    with gr.Tab("Coefficient Interpretation"):

        gr.Markdown(
            """
            ## Multiple Linear Regression Coefficients

            Bronx is the reference borough. Each borough coefficient
            represents the expected price difference compared with a
            similar Bronx listing, holding the other variables constant.
            """
        )

        gr.Dataframe(
            value=coefficient_table.round(4),
            label="Regression Coefficients",
            interactive=False
        )

        gr.Textbox(
            value=coefficient_interpretation,
            label="Business Interpretation",
            lines=20,
            interactive=False
        )

    # TAB 5: PRICE SIMULATOR
    with gr.Tab("Price Simulator"):

        gr.Markdown(
            """
            ## Potential Listing Price Simulator

            Enter the listing characteristics to estimate the expected
            nightly Airbnb price.
            """
        )

        simulator_borough = gr.Dropdown(
            choices=simulator_borough_choices,
            value="Manhattan",
            label="Borough"
        )

        with gr.Row():

            simulator_minimum_nights = gr.Number(
                value=3,
                minimum=1,
                label="Minimum Nights"
            )

            simulator_reviews = gr.Number(
                value=20,
                minimum=0,
                label="Number of Reviews"
            )

        with gr.Row():

            simulator_reviews_month = gr.Number(
                value=1.0,
                minimum=0,
                label="Reviews per Month"
            )

            simulator_availability = gr.Slider(
                minimum=0,
                maximum=365,
                value=180,
                step=1,
                label="Availability per Year"
            )

        prediction_button = gr.Button(
            "Estimate Nightly Price",
            variant="primary"
        )

        prediction_output = gr.Markdown()

        prediction_button.click(
            fn=predict_price,
            inputs=[
                simulator_borough,
                simulator_minimum_nights,
                simulator_reviews,
                simulator_reviews_month,
                simulator_availability
            ],
            outputs=prediction_output
        )

print("Gradio application created successfully.")

In [ ]:
app.queue()

app.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ae7a5d86b4b1354dfb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
